# Step 2 — Risk Scoring & Classification
Reads `clauses_raw.csv`, applies rule classification, risk scoring, NER, and explanation generation.
Outputs `risk_labeled_clauses.csv`, `conflict_registry.csv`, and `gap_analysis.csv`.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
from pathlib import Path

from src.semantic.rule_classifier import classify_all_clauses
from src.risk.risk_scorer import score_all_clauses
from src.risk.explainer import apply_explanations
from src.semantic.ner_extractor import run_ner_on_all_clauses

OUTPUT_PATH = Path('../app/outputs/risk_reports')
RAW_CSV     = OUTPUT_PATH / 'clauses_raw.csv'

if not RAW_CSV.exists():
    raise FileNotFoundError(f'Run step1_ingestion.ipynb first. Missing: {RAW_CSV}')

df = pd.read_csv(RAW_CSV)
print(f'Loaded {len(df)} clauses from {RAW_CSV}')
df.head()

Loaded 112 clauses from ..\app\outputs\risk_reports\clauses_raw.csv


,clause_id,section,clause_text,source,document,word_count
0,data_protection_policy.txt_0000,7.1 Designation,REGULATORY DOCUMENT: DATA PROTECTION AND PRIVA...,data_protection_policy.txt,data_protection_policy.txt,34
1,data_protection_policy.txt_0001,7.1 Designation,All entities subject to this regulation must c...,data_protection_policy.txt,data_protection_policy.txt,17
2,data_protection_policy.txt_0002,7.1 Designation,"""Personal data"" means any information relating...",data_protection_policy.txt,data_protection_policy.txt,13
3,data_protection_policy.txt_0003,7.1 Designation,"""Data controller"" refers to the entity that de...",data_protection_policy.txt,data_protection_policy.txt,16
4,data_protection_policy.txt_0004,7.1 Designation,"""Processing"" means any operation performed upo...",data_protection_policy.txt,data_protection_policy.txt,8


In [3]:
# Convert to list of dicts for pipeline
clauses = df.to_dict('records')

# 1. Rule-based classification
print('Classifying clauses...')
clauses = classify_all_clauses(clauses)

# 2. Risk scoring
print('Scoring risk...')
clauses = score_all_clauses(clauses)

# 3. Explanations
print('Generating explanations...')
clauses = apply_explanations(clauses)

print('Pipeline complete.')

Classifying clauses...
Scoring risk...
Generating explanations...
Pipeline complete.


In [4]:
# NER (optional — slow on large datasets)
print('Running NER...')
clauses = run_ner_on_all_clauses(clauses)
print('NER complete.')

Running NER...
NER complete.


In [5]:
# Save risk-labeled clauses
result_df = pd.DataFrame(clauses)

# Drop entities dict column for CSV (not CSV-friendly)
if 'entities' in result_df.columns:
    result_df = result_df.drop(columns=['entities'])

out_file = OUTPUT_PATH / 'risk_labeled_clauses.csv'
result_df.to_csv(out_file, index=False)
print(f'Saved: {out_file}')

# Summary
print('\nRisk distribution:')
print(result_df['risk_level'].value_counts())
print('\nClause type distribution:')
print(result_df['clause_type'].value_counts())
result_df.head()

Saved: ..\app\outputs\risk_reports\risk_labeled_clauses.csv

Risk distribution:
risk_level
LOW       80
MEDIUM    28
HIGH       4
Name: count, dtype: int64

Clause type distribution:
clause_type
obligation     64
prohibition    26
general        19
condition       3
Name: count, dtype: int64


,clause_id,section,clause_text,source,document,word_count,clause_type,risk_score,risk_level,has_penalty,has_deadline,has_law_reference,explanation
0,data_protection_policy.txt_0000,7.1 Designation,REGULATORY DOCUMENT: DATA PROTECTION AND PRIVA...,data_protection_policy.txt,data_protection_policy.txt,34,condition,15,LOW,False,True,False,This is a LOW risk clause of general advisory ...
1,data_protection_policy.txt_0001,7.1 Designation,All entities subject to this regulation must c...,data_protection_policy.txt,data_protection_policy.txt,17,obligation,20,LOW,False,False,False,This is a LOW risk clause of general advisory ...
2,data_protection_policy.txt_0002,7.1 Designation,"""Personal data"" means any information relating...",data_protection_policy.txt,data_protection_policy.txt,13,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...
3,data_protection_policy.txt_0003,7.1 Designation,"""Data controller"" refers to the entity that de...",data_protection_policy.txt,data_protection_policy.txt,16,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...
4,data_protection_policy.txt_0004,7.1 Designation,"""Processing"" means any operation performed upo...",data_protection_policy.txt,data_protection_policy.txt,8,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...


In [7]:
# Generate conflict registry (requires embeddings — placeholder approach)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.risk.similarity_engine import find_similar_pairs, find_gap_clauses

print('Computing TF-IDF similarity for conflict detection...')
texts = result_df['clause_text'].fillna('').tolist()

vec = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = vec.fit_transform(texts)
tfidf_dense  = tfidf_matrix.toarray()

clause_list = result_df.to_dict('records')
pairs = find_similar_pairs(clause_list, tfidf_dense, threshold=0.55)

conflict_df = pd.DataFrame(pairs)
conflict_file = OUTPUT_PATH / 'conflict_registry.csv'
conflict_df.to_csv(conflict_file, index=False)
print(f'Saved {len(conflict_df)} conflicts → {conflict_file}')

Computing TF-IDF similarity for conflict detection...
Saved 1 conflicts → ..\app\outputs\risk_reports\conflict_registry.csv


In [8]:
# Gap analysis
gaps = find_gap_clauses(clause_list, tfidf_dense, threshold=0.05)
gap_df = pd.DataFrame(gaps)
gap_file = OUTPUT_PATH / 'gap_analysis.csv'
gap_df.to_csv(gap_file, index=False)
print(f'Saved {len(gap_df)} gaps → {gap_file}')

Saved 0 gaps → ..\app\outputs\risk_reports\gap_analysis.csv
